# Aula 03 · Derivando dados medidos

Esta aula apresenta o [capítulo 3 do site](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/). A ideia central: **no
mundo real não há fórmula, só uma tabela** — e o $h$ é a distância entre as
medições. As fórmulas da Aula 02 continuam valendo, com três cuidados novos:
espaçamento irregular, as bordas da tabela e o **ruído**, que a derivada amplifica.

**Ao fim da aula você consegue:**

1. derivar uma tabela com espaçamento irregular, usando a central no meio e as
   fórmulas de um lado só nas bordas;
2. explicar, com uma conta no papel, por que derivar amplifica o ruído de um
   sensor;
3. reduzir o ruído afastando os vizinhos — e reconhecer quando isso apaga o
   fenômeno;
4. aplicar o mesmo cálculo a esporte, agricultura, física e saúde.

**Roteiro:** 🧩 · 1. a tabela · 2. as bordas · 3. 🧑‍🏫 ruído · 4. o elevador ·
5. outra área · 🎯 prática · 🧩 o alerta consertado · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

# --- dados desta aula (baixados do site, se ainda não estiverem aqui) ---
import os
import urllib.request

for ARQUIVO in ["bolt_berlim_2009.csv", "estufa_temperatura.csv", "elevador_lacouth.csv", "casos_acumulados.csv", "rio_nivel.csv"]:
    if not os.path.exists(ARQUIVO):
        urllib.request.urlretrieve("https://lacouth.github.io/metodos_telecom-site/dados/" + ARQUIVO, ARQUIVO)
    print(ARQUIVO, "pronto")

## 🧩 O problema da aula

> **Defesa Civil — hidrologia.**
>
> *Uma régua automática mede o nível de um rio a cada 15 minutos. O protocolo da
> Defesa Civil manda **emitir alerta de cheia quando o rio sobe mais de 30 cm por
> hora**. O técnico escreveu um programa que calcula essa taxa e, na última chuva
> forte, o alerta disparou **duas horas antes** de o rio começar a subir de
> verdade. A população evacuou à toa — e da próxima vez vai demorar a acreditar.
> "**O programa está errado? Como fazer um alerta em que dá para confiar?**"*

A taxa de subida é uma derivada — de uma tabela, não de uma fórmula. No fim da aula
você conserta o alerta.

## 1. Quando só temos a tabela

Os tempos de passagem de Usain Bolt a cada 10 m, no recorde mundial de Berlim
(2009), estão no arquivo `bolt_berlim_2009.csv`: a coluna 0 é a distância (m), a
coluna 1 é o tempo (s). Ninguém mediu a velocidade dele. Dá para calculá-la?

📖 [capítulo 3 · Quando só temos a tabela](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#quando-so-temos-a-tabela)

> 🧰 **Comando novo: `np.loadtxt` e as colunas `dados[:, j]`**
>
> Um arquivo **CSV** é uma tabela em texto: uma linha por medição, colunas separadas
> por vírgula, e a primeira linha com os nomes das colunas. O do Bolt começa assim:
>
> ```
> distancia_m,tempo_s
> 0,0.00
> 10,1.89
> ```
>
> `np.loadtxt("arquivo.csv", delimiter=",", skiprows=1)` lê o arquivo e devolve a
> tabela inteira como um array de linhas e colunas. `delimiter=","` diz que as
> colunas são separadas por vírgula; `skiprows=1` pula a primeira linha, o
> cabeçalho (texto não vira número).
>
> Para pegar uma **coluna**, use `dados[:, j]`: os dois-pontos querem dizer "todas
> as linhas", e `j` é o número da coluna, contando do zero. Cada coluna é um array
> comum, com índice, fatia e contas como na Aula 00. (O arquivo já está na pasta: a
> célula ⚙️ o baixou.)

In [ ]:
# 🧰 exemplo — só rode e veja a saída
dados = np.loadtxt("bolt_berlim_2009.csv", delimiter=",", skiprows=1)
print(dados[:3])
print(dados[:, 0])

**✍️ Passo 1.** Leia o arquivo com `np.loadtxt("bolt_berlim_2009.csv", delimiter=",", skiprows=1)`, guarde a coluna 0 em `x` e a coluna 1 em `t`, e imprima `t`.

In [ ]:
# ✍️ passo 1

**Preveja:** os intervalos de tempo entre uma marca e a seguinte são iguais?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: 1,89 s para os primeiros 10 m, 0,99 s para os seguintes, e cerca de
0,81 s no meio da prova. O $h$ (em tempo) **não é constante** — ele muda
de ponto para ponto.

</details>

**✍️ Passo 2.** A velocidade aos 60 m é a central com os vizinhos das posições 5 e 7. Calcule `(x[7] - x[5]) / (t[7] - t[5])` e imprima em m/s e em km/h (vezes 3,6).

In [ ]:
# ✍️ passo 2

**Preveja:** quanto você acha que Bolt corria no meio da prova, em km/h?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`12.27` m/s, ou **44,2 km/h**. Repare no denominador: é a diferença **real**
entre os tempos dos vizinhos, e não `2 * h` — não existe um `h` único.

</details>

**✍️ Passo 3.** Num laço `for i in range(1, len(t) - 1):`, imprima `x[i]` e a velocidade central em cada marca.

In [ ]:
# ✍️ passo 3

**Preveja:** Bolt foi mais rápido no fim da prova?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: o máximo é entre 60 e 70 m (12,27 m/s), e ele **perde velocidade** no
fim (12,05 m/s aos 90 m). Até o recordista mundial desacelera.

📖 [capítulo 3 · Quando só temos a tabela](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#quando-so-temos-a-tabela)

</details>

## 2. As bordas da tabela

No primeiro ponto não há vizinho à esquerda; no último, não há à direita. Nas
bordas, só dá para usar a **progressiva** (no início) e a **regressiva** (no fim) —
que são $O(h)$.

📖 [capítulo 3 · As bordas da tabela](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#as-bordas-da-tabela)

**✍️ Passo 4.** Calcule a velocidade no primeiro ponto (`t = 0`) com a progressiva: `(x[1] - x[0]) / (t[1] - t[0])`.

In [ ]:
# ✍️ passo 4

**Preveja:** no instante do tiro de largada, qual era a velocidade real de Bolt?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A conta dá **5,29 m/s** — mas no tiro de largada Bolt estava **parado**. A
progressiva mediu a velocidade *média* dos primeiros 10 m, e atribuiu ao
instante 0.

</details>

> ⚠️ **Armadilha.** A borda **não avisa** que está errada: 5,29 m/s parece um número razoável.
Nas pontas da tabela, desconfie sempre — o erro é maior, e a derivada da
derivada (a aceleração) acumula os dois erros.

### 🎯 Sua vez — A regressiva no fim

Escreva `ultima_taxa(t, y)`, que devolve a derivada de `y` no **último**
ponto da tabela, pela regressiva. As listas podem ter qualquer tamanho.

In [ ]:
def ultima_taxa(t, y):
    # sua solução aqui
    pass

In [ ]:
confere(ultima_taxa, [
    (([0, 2, 4], [0, 12, 48]), 18.0),
    (([0, 1, 3], [5, 5, 11]), 3.0),
    (([0, 0.5], [1, 2]), 2.0),
])

<details>
<summary><b>💡 Dica</b></summary>

O último índice é `len(t) - 1`, e o vizinho dele é `len(t) - 2`.

</details>

## 3. Derivar amplifica o ruído

Um sensor IoT numa estufa agrícola mede a temperatura a cada 5 minutos
(`estufa_temperatura.csv`: hora, °C). Todo sensor tem **ruído**: a leitura oscila
alguns décimos de grau em torno do valor real.

📖 [capítulo 3 · Derivar amplifica o ruído](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#derivar-amplifica-o-ruido)

### 🧑‍🏫 No quadro — por que o ruído cresce

Caderno de papel aberto. No quadro:

1. cada leitura é o valor verdadeiro mais um erro: $y_i = f(t_i) + \varepsilon_i$;
2. na central, a parte verdadeira dá a derivada, e sobra
   $\dfrac{\varepsilon_{i+1} - \varepsilon_{i-1}}{2h}$;
3. o numerador é do tamanho do ruído, **não encolhe com $h$**; o denominador sim;
4. com vizinhos a distância $k$, o denominador vira $2kh$: o ruído cai $k$ vezes,
   mas o erro de truncamento cresce.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$
\frac{y_{i+1} - y_{i-1}}{2h} = \underbrace{\frac{f(t_{i+1}) - f(t_{i-1})}{2h}}_{\approx f'(t_i)}
\; + \; \underbrace{\frac{\varepsilon_{i+1} - \varepsilon_{i-1}}{2h}}_{\text{ruído} \;\div\; 2h}
$$

Com ruído de $\pm 0{,}15$ °C e $2h = 10$ min $= 0{,}167$ h, o segundo termo chega
a cerca de $\pm 1$ °C/h — do tamanho da própria taxa que se quer medir. É o mesmo
cabo de guerra do $h$ pequeno demais da Aula 02, com o sensor no lugar do
arredondamento.

</details>

**✍️ Passo 5.** Leia `"estufa_temperatura.csv"` (pule 1 linha), guarde as colunas em `th` e `T`, e faça `plt.plot(th, T, ".")` e `plt.show()`.

In [ ]:
# ✍️ passo 5

**Preveja:** olhando o gráfico, a derivada desta curva vai ser suave ou irregular?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A curva **parece limpa**: um ciclo diário, mínimo de madrugada e máximo à
tarde. O ruído de 0,15 °C quase não aparece numa escala de 20 a 32 °C.

</details>

**✍️ Passo 6.** Num laço de `1` a `len(th) - 2`, calcule a central com os vizinhos imediatos, guarde numa lista `taxas` (com `append`) e imprima `max(taxas)`.

In [ ]:
# ✍️ passo 6

**Preveja:** a taxa máxima verdadeira do ciclo é 1,57 °C/h. Quanto vai dar?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Perto de **4 °C/h** — mais que o dobro da verdadeira. O ruído, invisível no
gráfico da temperatura, domina o gráfico da derivada.

</details>

**✍️ Passo 7.** Repita com vizinhos a distância `k = 6` (meia hora para cada lado): o laço vai de `k` a `len(th) - k - 1`, e a conta usa `i + k` e `i - k`.

In [ ]:
# ✍️ passo 7

**Preveja:** o máximo vai chegar mais perto de 1,57?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cai para perto de **1,9 °C/h**: bem mais perto. O denominador 6 vezes maior
divide o ruído por 6. Não chega exatamente a 1,57 — sobra um pouco de ruído.

📖 [capítulo 3 · Derivar amplifica o ruído](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#derivar-amplifica-o-ruido)

</details>

## 4. O elevador: um dado real

Agora um dado **real**: o acelerômetro de um celular dentro de um elevador,
50 leituras por segundo (`elevador_lacouth.csv`; pule **4** linhas de cabeçalho;
coluna 0 = tempo, coluna 3 = aceleração vertical). A derivada da aceleração é o
*jerk*, o "tranco" que o passageiro sente. Normas de conforto limitam o jerk a
1 ou 2 m/s³.

📖 [capítulo 3 · O elevador: um dado real](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#o-elevador-um-dado-real)

**✍️ Passo 8.** Leia o arquivo, guarde `te` (coluna 0) e `a` (coluna 3) e calcule o jerk com os vizinhos imediatos em todos os pontos do meio. Imprima o maior valor.

In [ ]:
# ✍️ passo 8

**Preveja:** o jerk vai ficar dentro do limite de conforto?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Passa de **67 m/s³** — mais de trinta vezes o limite. Nenhum passageiro
sentiria isso: é o ruído do sensor, dividido por $t_{i+1} - t_{i-1} \approx 0{,}04$ s.

</details>

**✍️ Passo 9.** Repita com `k = 25` (meio segundo para cada lado) e imprima o maior valor.

In [ ]:
# ✍️ passo 9

**Preveja:** e agora?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cerca de **2,9 m/s³**: o que um elevador real faz. O $k$ foi escolhido pela
**escala do fenômeno**: a partida e a parada duram uns 2 segundos, e meio
segundo para cada lado ainda "enxerga" as duas.

</details>

> ⚠️ **Armadilha.** Afastar **demais** também erra. Com `k = 250` (5 s), o máximo cai para
0,2 m/s³: o método passou por cima da partida e da parada, e apagou o
fenômeno junto com o ruído. Não existe $k$ certo para todo problema.

## 5. Mesmo método, outra área

Saúde pública: o boletim publica os casos **acumulados** de uma doença
(`casos_acumulados.csv`: dia, casos acumulados). Os casos novos de cada dia são a
derivada, com $h = 1$ dia: "acumulado de hoje menos o de ontem".

📖 [capítulo 3 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#mesmo-metodo-outra-area)

**✍️ Passo 10.** Leia o arquivo, calcule a lista de casos novos (regressiva, do dia 1 em diante) e imprima o dia com mais casos novos (`np.argmax` ajuda — cuidado com o +1).

In [ ]:
# ✍️ passo 10

**Preveja:** o modelo por trás dos dados tem o pico no dia 50. O dado vai dizer 50?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Diz dia **47**: a contagem diária tem ruído, e os dias 47 e 53 quase
empatam. É por isso que os boletins publicam a **média móvel de 7 dias** —
um remédio para ruído que você verá na Unidade 8.

</details>

### 🎯 Sua vez — Casos novos na semana

Escreva `novos_na_semana(acumulados, dia)`, que devolve quantos casos novos
surgiram **nos últimos 7 dias** até `dia` (inclusive). É uma regressiva com
$h = 7$ dias — só que sem dividir por 7.

In [ ]:
def novos_na_semana(acumulados, dia):
    # sua solução aqui
    pass

In [ ]:
acum = [0, 3, 7, 12, 20, 31, 45, 60, 80, 105]
confere(novos_na_semana, [
    ((acum, 7), 60),
    ((acum, 8), 77),
    ((acum, 9), 98),
])

<details>
<summary><b>💡 Dica</b></summary>

Acumulado do dia menos o acumulado de **sete dias antes**.

</details>

## 🎯 Prática

Vem do bloco *1. Quando só temos a tabela*.
📖 [capítulo 3 · Quando só temos a tabela](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#quando-so-temos-a-tabela)

### 🎯 Sua vez — Velocidade num ponto

Escreva `velocidade_kmh(t, x, i)`, que devolve a velocidade no ponto de
índice `i` (um ponto do meio da tabela), pela central com espaçamento
irregular, **em km/h**.

In [ ]:
def velocidade_kmh(t, x, i):
    # sua solução aqui
    pass

In [ ]:
confere(velocidade_kmh, [
    (([0, 1.89, 2.88, 3.78], [0, 10, 20, 30], 2), 38.095238095238095),
    (([0, 1, 2], [0, 5, 10], 1), 18.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Divida pela diferença **real** entre os tempos dos vizinhos, `t[i + 1] - t[i - 1]`, e multiplique por 3,6.

</details>

Vem do bloco *3. Derivar amplifica o ruído*.
📖 [capítulo 3 · Derivar amplifica o ruído](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/#derivar-amplifica-o-ruido)

### 🎯 Sua vez — A maior taxa

Escreva `maior_taxa(t, y, k)`, que devolve a **maior** derivada central da
tabela usando os vizinhos a distância `k`, entre todos os pontos que têm os
dois vizinhos.

In [ ]:
def maior_taxa(t, y, k):
    # sua solução aqui
    pass

In [ ]:
confere(maior_taxa, [
    (([0, 1, 2, 3, 4], [0, 1, 4, 9, 16], 1), 6.0),
    (([0, 1, 2, 3, 4], [0, 1, 4, 9, 16], 2), 4.0),
    (([0, 1, 2, 3], [5, 3, 1, 0], 1), -1.5),
])

<details>
<summary><b>💡 Dica</b></summary>

Padrão extremo: o primeiro candidato é a taxa do ponto `i = k`. O laço vai de
`k` até `len(t) - k - 1` — em Python, `range(k, len(t) - k)`.

</details>

## 🧩 Resolvendo o problema

> *"**O programa está errado? Como fazer um alerta em que dá para confiar?**"* — o
> técnico da Defesa Civil. O alerta dispara quando o rio sobe **mais de 0,30 m/h**.

A célula 📦 carrega a régua do rio (`t_rio` em horas, `nivel` em metros) e mostra o
gráfico.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Régua automática do rio: nível (m) a cada 15 minutos, durante uma cheia.
dados_rio = np.loadtxt("rio_nivel.csv", delimiter=",", skiprows=1)
t_rio = dados_rio[:, 0]        # hora do dia
nivel = dados_rio[:, 1]        # nível do rio (m)

plt.figure()
plt.plot(t_rio, nivel, ".-")
plt.xlabel("hora")
plt.ylabel("nível (m)")
plt.grid()
plt.show()

### 🎯 Sua vez — O primeiro alerta

Escreva `primeiro_alerta(t, nivel, k, limite)`, que percorre a tabela e
devolve a **hora** do primeiro ponto em que a derivada central com vizinhos
a distância `k` passa de `limite`. Se isso nunca acontecer, devolva `-1`.

In [ ]:
def primeiro_alerta(t, nivel, k, limite):
    # sua solução aqui
    pass

In [ ]:
confere(primeiro_alerta, [
    (([0, 1, 2, 3, 4, 5], [0, 0, 1, 3, 4, 4], 1, 1.2), 2),
    (([0, 1, 2, 3, 4, 5], [0, 0, 1, 3, 4, 4], 1, 0.4), 1),
    (([0, 1, 2, 3, 4, 5], [0, 0, 1, 3, 4, 4], 2, 1.2), -1),
])

<details>
<summary><b>💡 Dica</b></summary>

Um `for` de `k` até `len(t) - k - 1`; dentro dele, a taxa e um `if taxa > limite:`
com `return t[i]`. O `return -1` fica **depois** do laço, fora dele.

</details>

Agora, nos dados do rio, com vizinhos imediatos e com vizinhos mais afastados:

In [ ]:
for k in [1, 2, 4]:
    print(f"k = {k}: primeiro alerta às", primeiro_alerta(t_rio, nivel, k, 0.30), "h")

<details>
<summary><b>▶ O que os números dizem</b></summary>

Com `k = 1`, o alerta dispara às **6 h** — o alarme falso da história. Com `k = 4`
(uma hora para cada lado), dispara às **8h15**. O modelo por trás dos dados diz
que a subida passa de 0,30 m/h às 8h21: o alerta com `k = 4` acerta com 6 minutos de
antecedência.

O programa do técnico **não tinha erro de programação**: ele calculava a central
certinha. O erro era de método — derivar um dado com ruído com o menor $h$
possível. A resposta para a Defesa Civil tem duas partes: usar vizinhos a uma hora
de distância e, em troca, aceitar que o alerta "enxerga" a subida com uma hora de
janela.

</details>

## 📋 A lista, começada aqui

Abra a [Lista 03](https://lacouth.github.io/metodos_telecom-site/listas/lista03/). O **Exercício 01** é à mão (✏️): um reator químico
esfriando, com a temperatura a cada 5 minutos. Vamos fazê-lo juntos, no papel.

**a)** Em $t = 0$, $t = 10$ e $t = 20$ min, qual fórmula dá para usar em cada um?

<details>
<summary><b>▶ Resposta</b></summary>

Em $t = 0$ só existe o vizinho da direita: **progressiva**. Em $t = 10$ existem os
dois: **central**, dividindo por $10$ (a distância entre os vizinhos, e não 5). Em
$t = 20$, o último ponto: **regressiva**.

</details>

**b)** Antes de fazer as contas: os três resultados vão ter o mesmo sinal? Qual?

<details>
<summary><b>▶ Resposta</b></summary>

Negativo nos três: a temperatura só cai. Um sinal positivo em qualquer ponto é erro
de conta (quase sempre a ordem da subtração).

</details>

Termine o exercício e siga para o **Exercício 02**, a função `derivada_dados` que junta as três fórmulas.

## 🚪 Antes de sair

**1.** Um colega sugere resolver o alarme falso do rio medindo **mais vezes**: a
cada 1 minuto em vez de 15. Isso ajuda ou atrapalha a central com vizinhos
imediatos?

<details>
<summary><b>▶ Resposta da 1</b></summary>

**Atrapalha.** O ruído da régua continua de 5 cm, mas agora é dividido por 2 minutos
em vez de 30: a taxa falsa fica 15 vezes maior. Medir mais vezes só ajuda se você
**também** afastar os vizinhos (ou tirar médias) — mais dados, mesmo $h$ efetivo.

</details>

**2.** Por que a velocidade de Bolt calculada em $t = 0$ não é zero?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Porque no primeiro ponto só existe a progressiva, que mede a velocidade **média**
nos primeiros 10 m e a atribui ao instante 0. Nas bordas o erro é $O(h)$ — e o
$h$, aqui, é de quase 2 segundos.

</details>

**3.** Por que o programa do técnico passou em todos os testes dele e ainda assim deu alarme falso?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque ele testou com dados **sem ruído** (ou com uma fórmula). O código estava
certo; o método é que não servia para o dado real. Teste de método numérico precisa
de dado parecido com o de verdade.

</details>

## 🏠 Para casa

- Releia a seção sobre ruído do [capítulo 3](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/) e refaça a conta do 🧑‍🏫.
- Termine a [Lista 03](https://lacouth.github.io/metodos_telecom-site/listas/lista03/).
- A Unidade 3 começa com uma pergunta que ficou pendente duas vezes: **onde** a
  segunda derivada vale zero? Onde a taxa troca de sinal? É um problema de raiz.